# Bank Marketing - Classifiers
Machine Learning 
Assignment - 2
BITS ID : 2026AK05092

In [2]:
import os
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

os.makedirs("model", exist_ok=True)

## 1. Load data

In [4]:
TARGET = "y"
SEED = 7

train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")
print(train_df.shape, test_df.shape)
train_df.head()

(3616, 17) (905, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,49,self-employed,married,secondary,no,358,yes,yes,unknown,13,may,176,3,-1,0,unknown,no
1,50,technician,married,tertiary,no,199,yes,yes,cellular,3,feb,116,2,253,1,failure,no
2,32,services,single,secondary,no,182,yes,no,cellular,6,may,145,1,-1,0,unknown,no
3,54,services,married,secondary,no,158,yes,no,unknown,19,may,198,1,-1,0,unknown,no
4,37,blue-collar,married,secondary,no,140,yes,yes,cellular,7,may,490,1,-1,0,unknown,no


## 2. Split features and define preprocessing

In [6]:
X_train, y_train = train_df.drop(columns=TARGET), (train_df[TARGET] == "yes").astype(int)
X_test, y_test = test_df.drop(columns=TARGET), (test_df[TARGET] == "yes").astype(int)

cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
num_cols = X_train.select_dtypes(include="number").columns.tolist()
print("Numeric:", num_cols)
print("Categorical:", cat_cols)

def make_preprocessor():
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])

Numeric: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Categorical: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


## 3. Define and evaluate the five models

In [8]:
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=SEED),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": CalibratedClassifierCV(SVC(kernel="rbf", random_state=SEED), ensemble=False),
}

In [9]:
rows = []
for name, clf in classifiers.items():
    pipe = Pipeline([("prep", make_preprocessor()), ("clf", clf)])
    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    rows.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, pred),
        "AUC": roc_auc_score(y_test, proba),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "MCC": matthews_corrcoef(y_test, pred),
    })
    joblib.dump(pipe, f"model/{name.lower().replace(' ', '_')}.joblib")
    print("trained and saved:", name)

trained and saved: Logistic Regression
trained and saved: Decision Tree
trained and saved: KNN
trained and saved: Naive Bayes
trained and saved: Support Vector Machine


## 4. Comparison table

In [11]:
results = pd.DataFrame(rows).round(4)
results.to_csv("metrics_comparison.csv", index=False)
results

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.8972,0.8759,0.6222,0.2692,0.3758,0.3639
1,Decision Tree,0.8917,0.8060,0.5484,0.3269,0.4096,0.3686
2,KNN,0.8884,0.8112,0.5484,0.1635,0.2519,0.2560
3,Naive Bayes,0.8442,0.7734,0.3520,0.4231,0.3843,0.2976
4,Support Vector Machine,0.9017,0.8759,0.6744,0.2788,0.3946,0.3918


## 5. Saved Model files

In [13]:
print(sorted(os.listdir("model")))

['decision_tree.joblib', 'knn.joblib', 'logistic_regression.joblib', 'naive_bayes.joblib', 'support_vector_machine.joblib']
